In [19]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os
import numpy as np

In [15]:
# List files in the broadcast_logs directory
os.listdir("../../data/broadcast_logs/")

['BroadcastLogs_2018_Q3_M8_sample.CSV',
 'data dictionary.doc',
 'Call_Signs.csv',
 'ReferenceTables']

In [ ]:
# Create a Spark session
spark = SparkSession.builder.appName("Ch04").getOrCreate()

# Define the directory containing the data
DIRECTORY = "../../data/broadcast_logs/"

# Read the CSV file into a DataFrame
logs = spark.read.csv(
    os.path.join(DIRECTORY, "BroadcastLogs_2018_Q3_M8_sample.CSV"),
    sep = "|",
    header=True,
    inferSchema=True,
    timestampFormat="yyyy-MM-dd"
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/01 15:12:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Print the schema of the DataFrame
logs.printSchema()

root
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [ ]:
# Split the DataFrame columns into groups of ten and display the first five rows of each group
column_split = np.array_split(logs.columns, len(logs.columns) // 10)

for x in column_split:
    logs.select(*x).show(5, False)

+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+
|BroadcastLogID|LogServiceID|LogDate   |SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|
+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+
|1196192316    |3157        |2018-08-01|1         |4                  |NULL                  |13        |3              |3                |NULL            |
|1196192317    |3157        |2018-08-01|2         |NULL               |NULL                  |NULL      |1              |NULL             |NULL            |
|1196192318    |3157        |2018-08-01|3         |NULL               |NULL                  |NULL      |1              |NULL             |NULL            |
|1196192319    |3157        |2018-08-01|4         |NULL   

In [ ]:
# drop unnecessary columns
logs = logs.drop("BroadcastLogID", "SequenceNO")

In [ ]:
# Extract hours, minutes, and seconds from the Duration column
logs.select(F.col("Duration"),
            F.col("Duration").substr(1,2).cast("int").alias("dur_hours"),
            F.col("Duration").substr(4,2).cast("int").alias("dur_minutes"),
            F.col("Duration").substr(7,2).cast("int").alias("dur_seconds"),
).distinct().show(5, False)
            

+----------------+---------+-----------+-----------+
|Duration        |dur_hours|dur_minutes|dur_seconds|
+----------------+---------+-----------+-----------+
|00:04:52.0000000|0        |4          |52         |
|00:10:06.0000000|0        |10         |6          |
|00:26:41.0000000|0        |26         |41         |
|00:09:52.0000000|0        |9          |52         |
|00:04:26.0000000|0        |4          |26         |
+----------------+---------+-----------+-----------+
only showing top 5 rows



In [ ]:
# Create a new column "Duration_seconds" that converts the Duration into total seconds
logs = logs.withColumn(
    "Duration_seconds",
    (
        F.col("Duration").substr(1,2).cast("int") * 3600 +
        F.col("Duration").substr(4,2).cast("int") * 60 +
        F.col("Duration").substr(7,2).cast("int")
    ),
)

In [ ]:
# Display summary statistics for the Duration_seconds column
logs.describe("Duration_seconds").show()

+-------+------------------+
|summary|  Duration_seconds|
+-------+------------------+
|  count|            236724|
|   mean|124.30587942076004|
| stddev| 573.7742807594924|
|    min|                 1|
|    max|             23409|
+-------+------------------+



In [36]:
# Alternative method to display summary statistics
logs.select("Duration_seconds").summary().show()

+-------+------------------+
|summary|  Duration_seconds|
+-------+------------------+
|  count|            236724|
|   mean|124.30587942076004|
| stddev| 573.7742807594924|
|    min|                 1|
|    25%|                15|
|    50%|                15|
|    75%|                30|
|    max|             23409|
+-------+------------------+



In [ ]:
# Detailed summary statistics including specific percentiles
logs.select("Duration_seconds").summary("min", "10%", "50%", "90%", "max").show()

+-------+----------------+
|summary|Duration_seconds|
+-------+----------------+
|    min|               1|
|    10%|              15|
|    50%|              15|
|    90%|              30|
|    max|           23409|
+-------+----------------+

